# Sprint 3 — Inference: OULAD Risk Model (XGBoost)

Same pattern as `inference_uci.ipynb`: load the trained model, **save it as a
`.txt` file**, reload a fresh model *only* from that file, verify the
predictions match exactly, then run both batch and single-record inference.

## Same honesty check as the UCI notebook

`risk_model_oulad_v1.txt` is a fully valid, reloadable model — but it's binary
(UBJSON) content under a `.txt` name, not human-readable text. See
`inference_uci.ipynb`'s intro for the full explanation; Section 6 here again
provides a genuinely readable tree dump as a separate, non-reloadable
documentation file.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier

PROJECT_ROOT = r"C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli"
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

RISK_THRESHOLD = 0.527  # carried over from model_training_oulad.ipynb Section 7b
                         # (F1-optimal threshold, retrained on the updated 35-feature matrix)

## 1. Load the trained model and save it as `.txt`

In [2]:
joblib_path = os.path.join(MODELS_DIR, "risk_model_oulad_v1.joblib")
model = joblib.load(joblib_path)
print(f"Loaded from: {joblib_path}")
print(f"Model type: {type(model).__name__}")

txt_path = os.path.join(MODELS_DIR, "risk_model_oulad_v1.txt")
model.save_model(txt_path)
print(f"\nSaved as: {txt_path}")
print(f"File size: {os.path.getsize(txt_path):,} bytes")

Loaded from: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_oulad_v1.joblib
Model type: XGBClassifier

Saved as: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_oulad_v1.txt
File size: 418,130 bytes


c:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:36:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


## 2. Reload a fresh model *only* from the `.txt` file

In [3]:
loaded_model = XGBClassifier()
loaded_model.load_model(txt_path)
print("Model reloaded from .txt file.")

X_check = pd.read_csv(os.path.join(PROCESSED_DIR, "oulad_feature_matrix.csv")).drop(columns=["at_risk"]).head(20)
original_proba = model.predict_proba(X_check)[:, 1]
reloaded_proba = loaded_model.predict_proba(X_check)[:, 1]

max_diff = np.abs(original_proba - reloaded_proba).max()
print(f"Max prediction difference (.joblib vs .txt-reloaded): {max_diff:.10f}")
assert max_diff < 1e-6, "Reloaded model does not match the original"
print("Confirmed: identical predictions.")

Model reloaded from .txt file.
Max prediction difference (.joblib vs .txt-reloaded): 0.0000000000
Confirmed: identical predictions.


## 3. `predict_risk()`

In [4]:
def predict_risk(model, X: pd.DataFrame, threshold: float = RISK_THRESHOLD) -> pd.DataFrame:
    """Identical logic to model_training_oulad.ipynb's predict_risk()."""
    proba = model.predict_proba(X)[:, 1]

    def bucket(p):
        if p < 0.3:
            return "Low"
        elif p < 0.6:
            return "Medium"
        return "High"

    return pd.DataFrame(
        {
            "risk_score": proba.round(4),
            "predicted_label": np.where(proba >= threshold, "at_risk", "not_at_risk"),
            "risk_level": [bucket(p) for p in proba],
        },
        index=X.index,
    )

## 4. Batch inference — score every student

In [5]:
X_all = pd.read_csv(os.path.join(PROCESSED_DIR, "oulad_feature_matrix.csv")).drop(columns=["at_risk"])
batch_predictions = predict_risk(loaded_model, X_all)

print(f"Scored {len(batch_predictions):,} students using the .txt-reloaded model")
print(f"\nrisk_level distribution:")
print(batch_predictions["risk_level"].value_counts())
print(f"\nTop 10 highest-risk students:")
batch_predictions.sort_values("risk_score", ascending=False).head(10)

Scored 26,074 students using the .txt-reloaded model

risk_level distribution:
risk_level
Low       13006
High      10840
Medium     2228
Name: count, dtype: int64

Top 10 highest-risk students:


,risk_score,predicted_label,risk_level
3493,1.0,at_risk,High
2595,1.0,at_risk,High
8632,1.0,at_risk,High
20262,1.0,at_risk,High
16934,1.0,at_risk,High
5967,1.0,at_risk,High
12099,1.0,at_risk,High
18616,1.0,at_risk,High
5977,1.0,at_risk,High
12086,1.0,at_risk,High


## 5. Single-record inference — scoring one new (in-progress) student

Unlike UCI's static demographic snapshot, OULAD's features are **engagement
aggregates over a module's elapsed weeks** — meaning a realistic single-record
inference call would happen mid-course, using whatever engagement exists so
far (same idea as `risk_charts.ipynb`'s lead-time simulation, just as a single
lookup instead of a full week-by-week scan).

In [6]:
feature_columns = X_all.columns.tolist()
print("Required feature columns, in order:")
print(feature_columns)

Required feature columns, in order:
['total_clicks_sum', 'total_clicks_mean', 'total_clicks_max', 'days_active_sum', 'days_active_mean', 'num_weeks_active', 'engagement_trend', 'num_of_prev_attempts', 'studied_credits', 'pct_assessments_submitted', 'age_band_ordinal', 'highest_education_ordinal', 'imd_band_missing', 'imd_band_ordinal', 'code_module_BBB', 'code_module_CCC', 'code_module_DDD', 'code_module_EEE', 'code_module_FFF', 'code_module_GGG', 'region_East Midlands Region', 'region_Ireland', 'region_London Region', 'region_North Region', 'region_North Western Region', 'region_Scotland', 'region_South East Region', 'region_South Region', 'region_South West Region', 'region_Wales', 'region_West Midlands Region', 'region_Yorkshire Region', 'gender_M', 'disability_Y']


In [7]:
# Example: a student 10 weeks into a module (code_module 'BBB') with declining
# engagement and a couple of missed assessments -- the profile
# feature_engineering_oulad.ipynb's Section 8b boxplots associated with elevated risk.
new_student = {
    "total_clicks_sum": 145, "total_clicks_mean": 14.5, "total_clicks_max": 40,
    "days_active_sum": 22, "days_active_mean": 2.2, "num_weeks_active": 10,
    "engagement_trend": -9.5,  # declining: second half well below first half
    "num_of_prev_attempts": 0, "studied_credits": 60,
    "pct_assessments_submitted": 0.5,
    "age_band_ordinal": 0, "highest_education_ordinal": 1,
    "imd_band_missing": 0, "imd_band_ordinal": 35,
    "code_module_BBB": True, "code_module_CCC": False, "code_module_DDD": False,
    "code_module_EEE": False, "code_module_FFF": False, "code_module_GGG": False,
    "region_East Midlands Region": False, "region_Ireland": False, "region_London Region": True,
    "region_North Region": False, "region_North Western Region": False, "region_Scotland": False,
    "region_South East Region": False, "region_South Region": False, "region_South West Region": False,
    "region_Wales": False, "region_West Midlands Region": False, "region_Yorkshire Region": False,
    "gender_M": False, "disability_Y": False,
}

X_new = pd.DataFrame([new_student])[feature_columns]  # enforce exact training column order
result = predict_risk(loaded_model, X_new)
print(result.to_string(index=False))

 risk_score predicted_label risk_level
     0.6388         at_risk       High


## 6. Bonus — human-readable model dump

In [8]:
booster = loaded_model.get_booster()
dump_path = os.path.join(MODELS_DIR, "risk_model_oulad_v1_readable_dump.txt")
booster.dump_model(dump_path, with_stats=True)

print(f"Saved human-readable tree dump: {dump_path}")
with open(dump_path) as f:
    lines = f.readlines()
print(f"Total lines: {len(lines):,}")
print("\nFirst 12 lines (the first tree's top splits):")
print("".join(lines[:12]))

Saved human-readable tree dump: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_oulad_v1_readable_dump.txt
Total lines: 8,560

First 12 lines (the first tree's top splits):
booster[0]:
0:[pct_assessments_submitted<0.131578952] yes=1,no=2,missing=2,gain=10759.9316,cover=5364
	1:[pct_assessments_submitted<0.129032254] yes=3,no=4,missing=4,gain=20.3896484,cover=1832.37048
		3:[total_clicks_max<1766] yes=7,no=8,missing=8,gain=0.375488281,cover=1772.30664
			7:leaf=0.1986458,cover=1771.26245
			8:leaf=0.0532430373,cover=1.04419601
		4:[num_weeks_active<23] yes=9,no=10,missing=10,gain=37.8866577,cover=60.0638123
			9:[days_active_mean<1.38095236] yes=15,no=16,missing=16,gain=1.409729,cover=47.2397079
				15:leaf=0.0436845459,cover=1.55892801
				16:[num_weeks_active<22] yes=25,no=26,missing=26,gain=1.0324707,cover=45.6807823
					25:leaf=0.183925852,cover=42.5481911
					26:leaf=0.0790104419,cover=3.13258791



Not reloadable — same caveat as the UCI notebook. For a working model, use
`risk_model_oulad_v1.txt` via Section 1/2's `save_model()`/`load_model()`.

## 7. Summary

- **Saved** `models/risk_model_oulad_v1.txt` — binary content despite the name,
  same caveat as UCI's inference notebook.
- **Verified round-trip:** `.txt`-only reload matches the original `.joblib`
  exactly.
- **Batch inference** on the full updated (35-feature) matrix.
- **Single-record inference** — built from the exact training column order,
  using a mid-course "10 weeks in, declining engagement" example profile.
- **Bonus readable dump** kept clearly separate from the functional `.txt` model.